# 🏗️ Sistema Experto Legal — Cumplimiento Normativo de Obra
---
**Materia:** Sistemas Expertos  
**Descripción:** Sistema que evalúa el cumplimiento normativo de una obra en construcción en Argentina, verificando documentación requerida, vencimientos y generando alertas automáticas.

---
## 📋 Tabla de contenidos
1. [Módulo 1 — Instalación y dependencias](#modulo-1)
2. [Módulo 2 — Modelos de datos](#modulo-2)
3. [Módulo 3 — Motor de inferencia](#modulo-3)
4. [Módulo 4 — Capa de datos](#modulo-4)
5. [Módulo 5 — Reportes](#modulo-5)
6. [Módulo 6 — Ejecución principal](#modulo-6)


---
## Módulo 1 — Instalación y dependencias <a name="modulo-1"></a>
Instalación de librerías externas e importación de módulos estándar.


In [ ]:
# ── Instalación (solo la primera vez) ───────────────────────────────
!pip install -q prettytable colorama

In [ ]:
# ── Importaciones ────────────────────────────────────────────────────
import os
import datetime
import csv
import json
from datetime import timedelta
from enum import Enum
from io import StringIO
from prettytable import PrettyTable
from colorama import Fore, Style, init

init(autoreset=True)

# ── Google Drive (Colab) ─────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_MONTADO = True
except Exception:
    DRIVE_MONTADO = False

print("✅ Dependencias cargadas correctamente.")

---
## Módulo 2 — Modelos de datos <a name="modulo-2"></a>
Define las estructuras de datos del sistema:
- **`NivelAlerta`** — enum con los niveles de gravedad posibles
- **`Etapa`** — enum con las etapas de una obra
- **`Obra`** — representa una obra con todos sus atributos normativos
- **`ReglaNormativa`** — encapsula una regla con su condición, documento y nivel de alerta


In [ ]:
# ── Enumeraciones ────────────────────────────────────────────────────

class NivelAlerta(Enum):
    """Niveles de gravedad de una alerta normativa."""
    BAJO     = "Bajo"
    MODERADO = "Moderado"
    CRITICO  = "Critico"   # sin tilde para evitar problemas de encoding


class Etapa(Enum):
    """Etapas posibles de una obra en construcción."""
    INICIO       = "inicio"
    EXCAVACION   = "excavacion"
    CONSTRUCCION = "construccion"
    FINALIZACION = "finalizacion"


# Paleta de colores para consola por nivel de alerta
COLOR_CONSOLA = {
    NivelAlerta.CRITICO.value:  Fore.RED,
    NivelAlerta.MODERADO.value: Fore.YELLOW,
    NivelAlerta.BAJO.value:     Fore.GREEN,
}

print("✅ Enumeraciones definidas.")

In [ ]:
# ── Clase Obra ───────────────────────────────────────────────────────

class Obra:
    """
    Representa una obra en construcción con todos sus atributos
    relevantes para la evaluación normativa.

    Atributos:
        nombre              (str)      : Nombre identificatorio de la obra.
        ubicacion           (str)      : Jurisdicción — CABA, GBA, etc.
        m2                  (float)    : Superficie total construida.
        subsuelos           (int)      : Cantidad de subsuelos.
        trabajadores        (int)      : Cantidad de trabajadores en obra.
        maquinaria_pesada   (bool)     : Indica si se usa maquinaria pesada.
        obra_nueva          (bool)     : True si es obra nueva, False si es refacción.
        etapa               (str)      : Etapa actual según Etapa enum.
        fecha_inicio        (datetime) : Fecha de inicio de la obra.
        tipo_edificio       (str)      : Tipo de edificio (residencial, comercial, etc.).
        patrimonio_protegido(bool)     : True si el inmueble tiene protección patrimonial.
    """

    def __init__(self, nombre: str, ubicacion: str, m2: float, subsuelos: int,
                 trabajadores: int, maquinaria_pesada: bool, obra_nueva: bool,
                 etapa: str, fecha_inicio: datetime.datetime,
                 tipo_edificio: str = "residencial",
                 patrimonio_protegido: bool = False):

        self.nombre                = nombre
        self.ubicacion             = ubicacion.upper()
        self.m2                    = m2
        self.subsuelos             = subsuelos
        self.trabajadores          = trabajadores
        self.maquinaria_pesada     = maquinaria_pesada
        self.obra_nueva            = obra_nueva
        self.etapa                 = etapa.lower()
        self.fecha_inicio          = fecha_inicio
        self.dias_transcurridos    = (datetime.datetime.now() - fecha_inicio).days
        self.tipo_edificio         = tipo_edificio
        self.patrimonio_protegido  = patrimonio_protegido
        self.documentos_entregados = []
        self.historial             = []   # registro de auditoría

    def registrar_entrega(self, documento: str, dias_validez: int = None):
        """
        Registra un documento entregado.

        Args:
            documento   : Nombre del documento entregado.
            dias_validez: Días de vigencia desde hoy. None = sin vencimiento.
        """
        vencimiento = (datetime.datetime.now() + timedelta(days=dias_validez)
                       if dias_validez else None)
        self.documentos_entregados.append({
            "documento":         documento,
            "fecha_entrega":     datetime.datetime.now(),
            "fecha_vencimiento": vencimiento,
        })

    def registrar_auditoria(self, resultado: dict):
        """Guarda el resultado de una verificación con timestamp."""
        self.historial.append({
            "fecha":     datetime.datetime.now().strftime('%d/%m/%Y %H:%M'),
            "resultado": resultado,
        })

print("✅ Clase Obra definida.")

In [ ]:
# ── Clase ReglaNormativa ─────────────────────────────────────────────

class ReglaNormativa:
    """
    Encapsula una regla normativa: qué condición debe cumplirse,
    qué documento exige y cuál es el nivel de alerta si falta.

    Acepta una condición simple (lambda) o una lista de condiciones
    que se evalúan con AND lógico.

    Args:
        condiciones     : Lambda o lista de lambdas que reciben una Obra.
        documento       : Nombre del documento requerido.
        alerta          : Nivel de alerta (usar NivelAlerta.X.value).
        etapa_requerida : Etapa a partir de la cual aplica la regla.
        plazo_validez_dias: Días de vigencia del documento desde inicio de obra.
    """

    def __init__(self, condiciones, documento: str, alerta: str,
                 etapa_requerida: str, plazo_validez_dias: int = 365):

        self.condiciones     = ([condiciones] if callable(condiciones)
                                else condiciones)
        self.documento       = documento
        self.alerta          = alerta
        self.etapa_requerida = etapa_requerida.lower()
        self.plazo_validez   = plazo_validez_dias

    def aplica_a(self, obra: Obra) -> bool:
        """Retorna True si TODAS las condiciones se cumplen para la obra dada."""
        return all(c(obra) for c in self.condiciones)

print("✅ Clase ReglaNormativa definida.")

---
## Módulo 3 — Motor de inferencia <a name="modulo-3"></a>
`SistemaExpertoLegal` aplica las reglas normativas sobre una obra y determina:
- Qué documentos faltan según la etapa actual
- Qué documentos entregados están por vencer o vencidos

La jerarquía de etapas garantiza que solo se exijan documentos
de la etapa actual o anteriores.


In [ ]:
# ── Motor de inferencia ──────────────────────────────────────────────

class SistemaExpertoLegal:
    """
    Motor de inferencia del sistema experto.

    Administra dos tipos de reglas:
    - Generales : aplican a todas las obras.
    - Por ubicación: aplican solo a obras en una jurisdicción específica.
    """

    JERARQUIA_ETAPAS = {
        Etapa.INICIO.value:       1,
        Etapa.EXCAVACION.value:   2,
        Etapa.CONSTRUCCION.value: 3,
        Etapa.FINALIZACION.value: 4,
    }

    def __init__(self):
        self._reglas_generales         = []
        self._reglas_por_ubicacion     = {}

    # ── carga de reglas ──────────────────────────────────────────────

    def agregar_regla(self, regla: ReglaNormativa, ubicacion: str = "TODAS"):
        """
        Agrega una regla al sistema.

        Args:
            regla    : Instancia de ReglaNormativa.
            ubicacion: 'TODAS' para regla general, o código de jurisdicción (ej: 'CABA').
        """
        if ubicacion.upper() == "TODAS":
            self._reglas_generales.append(regla)
        else:
            key = ubicacion.upper()
            self._reglas_por_ubicacion.setdefault(key, []).append(regla)

    def cargar_csv(self, ruta: str):
        """Carga reglas desde un archivo CSV externo."""
        with open(ruta, newline='', encoding='utf-8') as f:
            self._parsear_csv(f)

    def cargar_csv_string(self, csv_string: str):
        """Carga reglas desde un string CSV embebido."""
        self._parsear_csv(StringIO(csv_string))

    def _parsear_csv(self, archivo):
        """Parsea el CSV y construye ReglaNormativa para cada fila."""
        reader = csv.DictReader(archivo)
        for row in reader:
            ubicacion = row['ubicacion'].strip().upper()
            doc       = row['documento'].strip()
            alerta    = row['alerta'].strip()
            etapa     = row['etapa'].strip().lower()
            plazo     = int(row['plazo_validez']) if row.get('plazo_validez','').strip() else 365
            campo     = row['condicion'].strip()
            operador  = row['operador'].strip()
            valor     = row['valor_condicion'].strip()

            cond_fn = self._construir_condicion(campo, operador, valor)
            regla   = ReglaNormativa(cond_fn, doc, alerta, etapa, plazo)
            self.agregar_regla(regla, ubicacion)

    @staticmethod
    def _construir_condicion(campo: str, operador: str, valor: str):
        """
        Construye una función lambda a partir de los campos del CSV.
        Soporta operadores: vacío (siempre True), '>' y '=='.
        """
        if not campo or not operador:
            return lambda o: True
        if operador == '>':
            if valor.replace('.', '', 1).isdigit():
                umbral = float(valor)
                return lambda o, c=campo, u=umbral: getattr(o, c, 0) > u
        elif operador == '==':
            if valor.lower() == 'true':
                return lambda o, c=campo: getattr(o, c, False) is True
            elif valor.lower() == 'false':
                return lambda o, c=campo: getattr(o, c, True) is False
            else:
                return lambda o, c=campo, v=valor: getattr(o, c, '') == v
        return lambda o: False

    # ── evaluación ───────────────────────────────────────────────────

    def _reglas_aplicables(self, obra: Obra) -> list:
        """Retorna todas las reglas cuya condición se cumple para la obra."""
        reglas = [r for r in self._reglas_generales if r.aplica_a(obra)]
        for r in self._reglas_por_ubicacion.get(obra.ubicacion, []):
            if r.aplica_a(obra):
                reglas.append(r)
        return reglas

    def verificar_faltantes(self, obra: Obra) -> list:
        """
        Determina qué documentos obligatorios faltan según la etapa actual.

        Returns:
            Lista de dicts con: documento, alerta, etapa_requerida,
            dias_restantes, accion.
        """
        nivel_actual = self.JERARQUIA_ETAPAS.get(obra.etapa, 0)
        entregados   = {d['documento'] for d in obra.documentos_entregados}
        faltantes    = []

        for regla in self._reglas_aplicables(obra):
            nivel_regla = self.JERARQUIA_ETAPAS.get(regla.etapa_requerida, 0)
            if nivel_actual >= nivel_regla and regla.documento not in entregados:
                vencimiento    = obra.fecha_inicio + timedelta(days=regla.plazo_validez)
                dias_restantes = (vencimiento - datetime.datetime.now()).days
                faltantes.append({
                    "documento":       regla.documento,
                    "alerta":          regla.alerta,
                    "etapa_requerida": regla.etapa_requerida,
                    "dias_restantes":  max(0, dias_restantes),
                    "accion":          f"Presentar antes del {vencimiento.strftime('%d/%m/%Y')}",
                })

        # Ordenar por gravedad: Crítico → Moderado → Bajo
        orden = {NivelAlerta.CRITICO.value: 0,
                 NivelAlerta.MODERADO.value: 1,
                 NivelAlerta.BAJO.value: 2}
        faltantes.sort(key=lambda x: orden.get(x['alerta'], 9))
        return faltantes

    def verificar_vencimientos(self, obra: Obra) -> list:
        """
        Detecta documentos entregados que vencen en los próximos 30 días
        o ya están vencidos.

        Returns:
            Lista de dicts con: documento, estado, dias_restantes, fecha_vencimiento.
        """
        hoy     = datetime.datetime.now()
        alertas = []
        for doc in obra.documentos_entregados:
            if doc['fecha_vencimiento'] is not None:
                dias = (doc['fecha_vencimiento'] - hoy).days
                if dias <= 30:
                    alertas.append({
                        "documento":         doc['documento'],
                        "dias_restantes":    max(0, dias),
                        "fecha_vencimiento": doc['fecha_vencimiento'].strftime('%d/%m/%Y'),
                        "estado":            "VENCIDO" if dias < 0 else "POR VENCER",
                    })
        return alertas

print("✅ Motor de inferencia definido.")

---
## Módulo 4 — Capa de datos <a name="modulo-4"></a>
Contiene los datos de normativas y la obra de ejemplo.  
En producción estos datos se cargarían desde archivos externos en Google Drive.

### Estructura del CSV de normativas
| Campo | Descripción |
|---|---|
| `ubicacion` | TODAS, CABA, GBA, etc. |
| `condicion` | Atributo de Obra a evaluar (vacío = siempre aplica) |
| `valor_condicion` | Valor a comparar |
| `operador` | `>` o `==` (vacío = siempre True) |
| `documento` | Nombre del documento requerido |
| `alerta` | Critico / Moderado / Bajo |
| `etapa` | inicio / excavacion / construccion / finalizacion |
| `plazo_validez` | Días de vigencia desde el inicio de obra |


In [ ]:
# ── Normativas embebidas (fallback) ──────────────────────────────────
# Usadas cuando no se encuentran los archivos externos en Drive.
# Para agregar nuevas reglas, añadir una fila al CSV sin tocar el código.

CSV_NORMATIVAS = """ubicacion,condicion,valor_condicion,operador,documento,alerta,etapa,plazo_validez
TODAS,,,,"Plano aprobado",Critico,inicio,30
TODAS,,,,"Contrato ART",Critico,inicio,365
TODAS,,,,"DNI Propietario",Bajo,inicio,365
TODAS,,,,"Libro de Obra",Moderado,inicio,365
TODAS,trabajadores,0,>,"Registro de Trabajadores (Ley 22.250)",Moderado,construccion,365
TODAS,maquinaria_pesada,True,==,"Habilitacion de maquinaria",Moderado,construccion,365
TODAS,obra_nueva,True,==,"Certificado ambiental",Moderado,construccion,365
CABA,m2,1000,>,"Permiso municipal",Critico,inicio,180
CABA,m2,1000,>,"Impacto ambiental",Moderado,inicio,365
CABA,patrimonio_protegido,True,==,"Permiso patrimonio historico",Critico,inicio,90
GBA,,,,"Permiso GBA",Moderado,inicio,365
GBA,subsuelos,0,>,"Estudio de suelos",Moderado,excavacion,180
"""

# ── Obra de ejemplo ───────────────────────────────────────────────────
JSON_OBRA = """
{
    "nombre": "Edificio Torres del Sur",
    "ubicacion": "CABA",
    "m2": 1200,
    "subsuelos": 2,
    "trabajadores": 10,
    "maquinaria_pesada": true,
    "obra_nueva": true,
    "etapa": "excavacion",
    "dias_desde_inicio": 10,
    "tipo_edificio": "residencial",
    "patrimonio_protegido": true
}
"""

print("✅ Datos embebidos cargados.")

In [ ]:
# ── Cargador de datos ────────────────────────────────────────────────

def cargar_sistema_y_obra(sistema: SistemaExpertoLegal,
                           ruta_csv: str, ruta_json: str,
                           csv_fallback: str, json_fallback: str):
    """
    Intenta cargar normativas y datos de obra desde archivos externos.
    Si no los encuentra, usa los datos embebidos como fallback.

    Args:
        sistema       : Instancia de SistemaExpertoLegal a configurar.
        ruta_csv      : Ruta al CSV de normativas en Drive.
        ruta_json     : Ruta al JSON de la obra en Drive.
        csv_fallback  : String CSV embebido (fallback).
        json_fallback : String JSON embebido (fallback).

    Returns:
        Instancia de Obra configurada.
    """
    try:
        if not os.path.exists(ruta_csv) or not os.path.exists(ruta_json):
            raise FileNotFoundError
        sistema.cargar_csv(ruta_csv)
        with open(ruta_json, 'r', encoding='utf-8') as f:
            datos = json.load(f)
        print(f"{Fore.GREEN}✅ Datos cargados desde archivos externos.")
    except (FileNotFoundError, Exception):
        sistema.cargar_csv_string(csv_fallback)
        datos = json.loads(json_fallback)
        print(f"{Fore.YELLOW}⚠️  Archivos no encontrados. Usando datos embebidos.")

    fecha_inicio = datetime.datetime.now() - timedelta(days=datos.pop('dias_desde_inicio'))
    return Obra(**datos, fecha_inicio=fecha_inicio)


print("✅ Cargador de datos definido.")

---
## Módulo 5 — Reportes <a name="modulo-5"></a>
Funciones de presentación de resultados:
- `imprimir_reporte()` — reporte en consola con colores y tabla
- `exportar_html()` — reporte visual descargable en HTML
- `recomendacion_legal()` — evaluación final del estado normativo


In [ ]:
# ── Helpers de comparación ───────────────────────────────────────────

def normalizar_alerta(s: str) -> str:
    """Normaliza un string de alerta para comparación sin dependencia de encoding."""
    return (s.strip().lower()
             .replace('í','i').replace('ó','o').replace('é','e'))


def es_critico(alerta: str) -> bool:
    return normalizar_alerta(alerta) == 'critico'


def es_moderado(alerta: str) -> bool:
    return normalizar_alerta(alerta) == 'moderado'

print("✅ Helpers definidos.")

In [ ]:
# ── Reporte en consola ───────────────────────────────────────────────

def imprimir_reporte(obra: Obra, faltantes: list, vencimientos: list):
    """
    Imprime el reporte de cumplimiento normativo en consola.

    Args:
        obra        : Obra evaluada.
        faltantes   : Lista de documentos faltantes (de verificar_faltantes).
        vencimientos: Lista de vencimientos (de verificar_vencimientos).
    """
    SEP = "="*70

    # Encabezado
    print(f"\n{SEP}")
    print("   SISTEMA EXPERTO LEGAL — CUMPLIMIENTO NORMATIVO DE OBRA")
    print(SEP)
    print(f"   Obra        : {obra.nombre}")
    print(f"   Ubicacion   : {obra.ubicacion}")
    print(f"   Etapa actual: {obra.etapa.capitalize()}")
    print(f"   Dias activa : {obra.dias_transcurridos}")
    print(SEP)

    # Documentacion faltante
    if faltantes:
        print("\n[1] DOCUMENTACION FALTANTE (obligatoria)")
        tabla = PrettyTable()
        tabla.field_names = ["Documento","Alerta","Etapa requerida","Dias restantes","Accion"]
        tabla.align = "l"
        for doc in faltantes:
            color = COLOR_CONSOLA.get(doc['alerta'], '')
            tabla.add_row([
                doc['documento'],
                color + doc['alerta'] + Style.RESET_ALL,
                doc['etapa_requerida'].capitalize(),
                doc['dias_restantes'],
                doc['accion'],
            ])
        print(tabla)
    else:
        print(f"\n{Fore.GREEN}✅ No hay documentacion faltante para la etapa actual.")

    # Vencimientos
    if vencimientos:
        print("\n[2] VENCIMIENTOS DE DOCUMENTOS ENTREGADOS")
        tabla_v = PrettyTable()
        tabla_v.field_names = ["Documento","Estado","Dias restantes","Fecha vencimiento"]
        tabla_v.align = "l"
        for v in vencimientos:
            color = Fore.RED if v['estado'] == "VENCIDO" else Fore.YELLOW
            tabla_v.add_row([
                v['documento'],
                color + v['estado'] + Style.RESET_ALL,
                v['dias_restantes'],
                v['fecha_vencimiento'],
            ])
        print(tabla_v)
    else:
        print(f"\n{Fore.GREEN}✅ Todos los documentos entregados estan vigentes.")

print("✅ Funcion imprimir_reporte definida.")

In [ ]:
# ── Recomendacion legal final ─────────────────────────────────────────

def recomendacion_legal(faltantes: list, vencimientos: list):
    """
    Evalua el estado normativo global y emite una recomendacion.

    Criterios:
    - ALERTA MAXIMA : hay documentos criticos faltantes o vencidos.
    - PRECAUCION    : hay plazos ajustados (menos de 30 dias).
    - OK            : todo en orden.

    Args:
        faltantes   : Lista de documentos faltantes.
        vencimientos: Lista de vencimientos proximos.
    """
    SEP = "="*70
    print(f"\n{SEP}")
    print("RECOMENDACION LEGAL AUTOMATICA")

    criticos_faltantes = any(es_critico(d['alerta']) for d in faltantes)
    criticos_vencidos  = any(v['estado'] == "VENCIDO" for v in vencimientos)
    plazos_ajustados   = (any(d['dias_restantes'] < 30 for d in faltantes) or
                          any(v['dias_restantes'] <= 15 for v in vencimientos))

    if criticos_faltantes or criticos_vencidos:
        print(Fore.RED +
              "ALERTA MAXIMA: Documentos criticos vencidos o sin presentar. "
              "Riesgo de paralizacion de obra.")
    elif plazos_ajustados:
        print(Fore.YELLOW +
              "PRECAUCION: Plazos ajustados. Gestione con urgencia para evitar multas.")
    else:
        print(Fore.GREEN +
              "La obra cumple con la normativa vigente. Puede continuar.")
    print(SEP)

print("✅ Funcion recomendacion_legal definida.")

In [ ]:
# ── Exportacion HTML ─────────────────────────────────────────────────

def exportar_html(obra: Obra, faltantes: list, vencimientos: list,
                  ruta: str = "reporte_obra.html"):
    """
    Genera un reporte HTML estilizado y lo guarda en disco.
    En Colab ofrece descarga automatica.

    Args:
        obra        : Obra evaluada.
        faltantes   : Lista de documentos faltantes.
        vencimientos: Lista de vencimientos proximos.
        ruta        : Nombre del archivo HTML de salida.
    """
    COLOR_HTML = {
        NivelAlerta.CRITICO.value:  "#dc3545",
        NivelAlerta.MODERADO.value: "#fd7e14",
        NivelAlerta.BAJO.value:     "#28a745",
    }

    def _filas_faltantes():
        if not faltantes:
            return ('<tr><td colspan="5" style="color:#155724;background:#d4edda">'
                    '✅ Sin documentacion faltante</td></tr>')
        rows = ""
        for d in faltantes:
            c = COLOR_HTML.get(d['alerta'], '#6c757d')
            rows += (f'<tr><td>{d["documento"]}</td>'
                     f'<td style="color:{c};font-weight:bold">{d["alerta"]}</td>'
                     f'<td>{d["etapa_requerida"].capitalize()}</td>'
                     f'<td>{d["dias_restantes"]}</td>'
                     f'<td>{d["accion"]}</td></tr>')
        return rows

    def _filas_vencimientos():
        if not vencimientos:
            return ('<tr><td colspan="4" style="color:#155724;background:#d4edda">'
                    '✅ Todos los documentos vigentes</td></tr>')
        rows = ""
        for v in vencimientos:
            c = "#dc3545" if v['estado'] == "VENCIDO" else "#fd7e14"
            rows += (f'<tr><td>{v["documento"]}</td>'
                     f'<td style="color:{c};font-weight:bold">{v["estado"]}</td>'
                     f'<td>{v["dias_restantes"]}</td>'
                     f'<td>{v["fecha_vencimiento"]}</td></tr>')
        return rows

    html = f"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <title>Reporte Legal — {obra.nombre}</title>
  <style>
    body     {{ font-family: Arial, sans-serif; margin: 40px; background: #f8f9fa; }}
    .header  {{ background: #343a40; color: white; padding: 20px 30px; border-radius: 8px; margin-bottom: 30px; }}
    h2       {{ color: #343a40; border-bottom: 2px solid #dee2e6; padding-bottom: 6px; }}
    table    {{ width: 100%; border-collapse: collapse; background: white;
                box-shadow: 0 1px 4px rgba(0,0,0,.1); border-radius: 6px; overflow: hidden; margin-bottom: 30px; }}
    th       {{ background: #495057; color: white; padding: 10px 14px; text-align: left; }}
    td       {{ padding: 8px 14px; border-bottom: 1px solid #dee2e6; }}
    tr:hover {{ background: #f1f3f5; }}
    .footer  {{ font-size: 12px; color: #6c757d; text-align: right; }}
  </style>
</head>
<body>
  <div class="header">
    <h1 style="margin:0">Sistema Experto Legal — Cumplimiento Normativo</h1>
    <p style="margin:8px 0 0">
      <strong>Obra:</strong> {obra.nombre} &nbsp;|&nbsp;
      <strong>Ubicacion:</strong> {obra.ubicacion} &nbsp;|&nbsp;
      <strong>Etapa:</strong> {obra.etapa.capitalize()} &nbsp;|&nbsp;
      <strong>Dias desde inicio:</strong> {obra.dias_transcurridos}
    </p>
  </div>
  <h2>Documentacion Faltante</h2>
  <table>
    <tr><th>Documento</th><th>Alerta</th><th>Etapa Requerida</th><th>Dias Restantes</th><th>Accion</th></tr>
    {_filas_faltantes()}
  </table>
  <h2>Vencimientos de Documentos Entregados</h2>
  <table>
    <tr><th>Documento</th><th>Estado</th><th>Dias Restantes</th><th>Fecha Vencimiento</th></tr>
    {_filas_vencimientos()}
  </table>
  <div class="footer">Reporte generado el {datetime.datetime.now().strftime('%d/%m/%Y %H:%M')}</div>
</body>
</html>"""

    with open(ruta, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f"{Fore.GREEN}✅ Reporte HTML guardado: {ruta}")

    try:
        from google.colab import files
        files.download(ruta)
    except Exception:
        pass

print("✅ Funcion exportar_html definida.")

---
## Módulo 6 — Ejecución principal <a name="modulo-6"></a>
Orquesta todos los módulos anteriores:
1. Inicializa el sistema
2. Carga los datos
3. Registra documentos entregados
4. Agrega reglas con múltiples condiciones (opcional)
5. Evalúa y genera reportes


In [ ]:
# ── 6.1 Inicializar sistema y cargar datos ───────────────────────────

sistema = SistemaExpertoLegal()

RUTA_CSV  = ('/content/drive/MyDrive/TP-Grupal-Jueves/normativas.csv'
             if DRIVE_MONTADO else 'normativas.csv')
RUTA_JSON = ('/content/drive/MyDrive/TP-Grupal-Jueves/proyecto_obra.json'
             if DRIVE_MONTADO else 'proyecto_obra.json')

obra = cargar_sistema_y_obra(
    sistema,
    ruta_csv    = RUTA_CSV,
    ruta_json   = RUTA_JSON,
    csv_fallback = CSV_NORMATIVAS,
    json_fallback = JSON_OBRA,
)

In [ ]:
# ── 6.2 Registrar documentos entregados ─────────────────────────────
# Agregar aqui los documentos que ya fueron presentados en la obra.

obra.registrar_entrega("Plano aprobado",  dias_validez=30)
obra.registrar_entrega("Contrato ART",    dias_validez=365)
obra.registrar_entrega("DNI Propietario", dias_validez=365)

print(f"{Fore.CYAN}📁 Documentos registrados: {len(obra.documentos_entregados)}")

In [ ]:
# ── 6.3 Reglas con multiples condiciones (opcional) ──────────────────
# Ejemplo: CABA + patrimonio protegido + mas de 500m2

regla_especial = ReglaNormativa(
    condiciones=[
        lambda o: o.ubicacion == "CABA",
        lambda o: o.patrimonio_protegido is True,
        lambda o: o.m2 > 500,
    ],
    documento        = "Inspeccion especial patrimonio historico",
    alerta           = NivelAlerta.CRITICO.value,
    etapa_requerida  = Etapa.INICIO.value,
    plazo_validez_dias = 60,
)
sistema.agregar_regla(regla_especial)

print("✅ Reglas adicionales cargadas.")

In [ ]:
# ── 6.4 Evaluacion ───────────────────────────────────────────────────

faltantes    = sistema.verificar_faltantes(obra)
vencimientos = sistema.verificar_vencimientos(obra)

# Guardar en historial de auditoria
obra.registrar_auditoria({
    "faltantes":    len(faltantes),
    "vencimientos": len(vencimientos),
})

In [ ]:
# ── 6.5 Reporte en consola ───────────────────────────────────────────

imprimir_reporte(obra, faltantes, vencimientos)
recomendacion_legal(faltantes, vencimientos)

In [ ]:
# ── 6.6 Exportar HTML ────────────────────────────────────────────────

exportar_html(obra, faltantes, vencimientos)

In [ ]:
# ── 6.7 Historial de auditoria ───────────────────────────────────────

print("\nHistorial de verificaciones:")
for entrada in obra.historial:
    r = entrada['resultado']
    print(f"  [{entrada['fecha']}] "
          f"Faltantes: {r['faltantes']} | "
          f"Vencimientos: {r['vencimientos']}")